# Experiment 007: MC-SFT control, the same-data rival to GRPO (Colab)

Supervised training on the exact 245 mmlu prompts grpo used, gold letters as
targets with a fixed neutral think sentence. Answers whether cheap studying of
the answer key reproduces the grpo gain. Exploratory, the registered pipeline
stays untouched.

The sweep: 3 exposures (2 / 8 / 32 passes) x 3 seeds x both scales, probe-only
evals so each run is minutes. Then full generative evals for the seed-0
checkpoints at every exposure, thats where the hours go. ~5.5h total on an a100.

One caveat to remember when reading the curve: each exposure point runs its own
linear lr decay over its own step count (the hf trainer default), so this is a
training-budget curve under the method as practised, not a pure exposure-only
isolation. Generative base references come from experiments 005 (0.5b) and 004
(3b), the 007 summaries keep base_acc null on purpose.


## 0. Compute units and mount Drive

Fill `cu_before` now. The open record is saved straight after Drive mounts,
so a disconnected session can be closed when the next runtime starts.


In [13]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 504.26   # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '007_colab_mc_sft', 'colab_cu_before': cu_before}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260717_160352 | cu_before = 504.26
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved open session: 20260717_160352
drive mounted -> /content/drive/MyDrive/inm363


## 1. Check the GPU


In [14]:
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), '| torch:', torch.__version__)


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-3d3a4155-effb-c7d0-c5ab-50dc9ae033d7)
cuda: True | torch: 2.11.0+cu128


## 2. Clone the repo (automatic)

This cell clones the main branch of the public repository without asking for a GitHub token. For online experiment tracking, add `WANDB_API_KEY` under the key icon in Colab Secrets and grant this notebook access. If using a private fork, set `REPO_SLUG` to its owner and name, then add a read-only `GH_TOKEN` for that fork. A missing token for a private fork triggers a one-time prompt.


In [15]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


fatal: destination path 'INM363-IndividualProject-Dissertation' already exists and is not an empty directory.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

Keep colabs own cuda torch, just pin the libraries that drifted.


In [16]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


deps installed


## 4. W&B key (automatic)

Reads `WANDB_API_KEY` from Colab Secrets and writes it to the file the code expects.
If the secret isn't set, it falls back to a manual upload of `wandb_api_key.txt`.


In [17]:
import os

# 1) Use the key file when it is already present.
# 2) Else read WANDB_API_KEY from colab secrets and write the file.
# 3) Else fall back to a manual upload.
if os.path.exists('wandb_api_key.txt'):
    print('wandb_api_key.txt already present, using it')
else:
    wkey = None
    try:
        from google.colab import userdata
        wkey = userdata.get('WANDB_API_KEY')
    except Exception:
        wkey = None
    if wkey:
        open('wandb_api_key.txt', 'w').write(wkey.strip())
        print('wrote wandb key from colab secret')
    else:
        from google.colab import files
        files.upload()  # pick wandb_api_key.txt
print('have key file:', os.path.exists('wandb_api_key.txt'))


wandb_api_key.txt already present, using it
have key file: True


## 5. Check the Drive record

Outputs (results + checkpoints) are written straight to drive via `output_root`
in the config, so they survive a colab disconnect and a crashed run can resume.


In [18]:
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')


open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260717_160352.json


## 6. Build the data splits

Materialises the mmlu splits and runs the leakage check. The same 245 deduped
dev+val prompts feed the mc-sft targets here instead of grpo rollouts.


In [19]:
!python scripts/build_data.py


dropped 1 grpo prompt(s) that also sit in the test split:
  [high_school_psychology] 'Answer the multiple choice question. Think through it step by step inside <think></think> tags, then'...
sft rows:  7473
grpo rows: 245  by tier: {'easy': 110, 'medium': 82, 'hard': 53}
eval rows: 1884  by tier: {'easy': 879, 'medium': 609, 'hard': 396}
no leakage, three splits ready


## 7. Wiring check

Quick sanity that the config loads, the gpu is seen, and the trackers resolve.


In [20]:
!python scripts/run.py --config configs/007_p2_colab_b05.yaml


env=COLAB mode=FULL scale=B05 condition=SFT_ONLY
model=Qwen/Qwen2.5-0.5B device=cuda stages=['base_eval', 'sft', 'sft_eval']
steps=31 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/007_p2_COLAB_FULL_B05_sft_only_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/007_p2_COLAB_FULL_B05_sft_only_seed0/
local smoke wiring ok


## 8. The training sweep (probe-only)

18 small sft runs: 3 exposures x 3 seeds x both scales. No grpo stage at all,
and generative evals are skipped here (probe only), so each run takes minutes.
Base probes are shared per exposure id, seeds reuse them.

**If colab disconnects:** re-run the cell, finished runs skip in seconds.


In [21]:
# The whole training sweep, probe-only. Python loop with ! calls so the logs
# stream live, a bash cell would buffer everything until the end and look frozen
sweep = ['007_p2_colab_b05', '007_p8_colab_b05', '007_p32_colab_b05',
         '007_p2_colab_b3', '007_p8_colab_b3', '007_p32_colab_b3']
for c in sweep:
    for s in (0, 1, 2):
        print(f'===== {c} seed {s} =====')
        !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{c}.yaml --seed {s} --stage all


Streaming output truncated to the last 5000 lines.
wandb: ⣯ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⣟ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⡿ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⢿ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⣻ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⣽ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⣾ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⣷ uploading history steps 68-112, summary, console lines 69-113 (28s)
wandb: ⣯ uploading history steps 68-112, summary, console lines 69-113 (29s)
wandb: ⣟ uploading history steps 68-112, summary, console lines 69-113 (29s)
wandb: ⡿ uploading history steps 68-112, summary, console lines 69-113 (29s)
wandb: ⢿ uploading history steps 68-112, summary, console lines 69-113 (29s)
wandb: ⣻ uploading histor

## 9. Generative evals (seed 0, all exposures)


In [22]:
# Generative evals for the seed-0 checkpoint at every exposure, both scales.
# Same live-streaming loop style as the sweep above
D = '/content/drive/MyDrive/inm363'
for prefix, suffix, sc in [('p2', 'b05', 'B05'), ('p8', 'b05', 'B05'), ('p32', 'b05', 'B05'),
                           ('p2', 'b3', 'B3'), ('p8', 'b3', 'B3'), ('p32', 'b3', 'B3')]:
    tag = f'007_{prefix}_COLAB_FULL_{sc}_sft_only_seed0'
    print(f'===== gen eval {tag} =====')
    !python scripts/eval_adapter.py --config configs/007_{prefix}_colab_{suffix}.yaml --adapter {D}/checkpoints/{tag}/sft --condition sft --out {D}/results/{tag}


===== gen eval 007_p2_COLAB_FULL_B05_sft_only_seed0 =====
Loading weights: 100% 290/290 [00:00<00:00, 4107.97it/s]
sft eval (1884 items): 100% 118/118 [04:51<00:00,  2.47s/it, acc 0.442 fmt 0.509]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.5176336746302617, 'medium': 0.4318555008210181, 'hard': 0.2904040404040404}
probe per-tier: {'easy': 0.6882821387940842, 'medium': 0.5221674876847291, 'hard': 0.351010101010101}
===== gen eval 007_p8_COLAB_FULL_B05_sft_only_seed0 =====
Loading weights: 100% 290/290 [00:00<00:00, 6096.71it/s]
sft eval (1884 items): 100% 118/118 [04:12<00:00,  2.14s/it, acc 0.538 fmt 0.983]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.6666666666666666, 'medium': 0.4614121510673235, 'hard': 0.3712121212121212}
probe per-tier: {'easy': 0.6734926052332195, 'medium': 0.5106732348111659, 'hard': 0.35353535353535354}
===== gen eval 007_p32_COLAB_FULL_B05_sft_only_seed0 =====

## 10. Peek at the results

Probe accuracies for all 18 runs, generative where already evaluated.


In [23]:
import json, os

# probe numbers for every run, gen numbers where eval_adapter has been run
D = '/content/drive/MyDrive/inm363'
for p in ('p2', 'p8', 'p32'):
    for sc in ('B05', 'B3'):
        for s in (0, 1, 2):
            tag = f'007_{p}_COLAB_FULL_{sc}_sft_only_seed{s}'
            d = os.path.join(D, 'results', tag)
            lp_f = os.path.join(d, 'sft_logprob.json')
            ev_f = os.path.join(d, 'sft_eval.json')
            lp = json.load(open(lp_f))['per_tier_acc'] if os.path.exists(lp_f) else None
            ev = json.load(open(ev_f))['per_tier_acc'] if os.path.exists(ev_f) else None
            fmt = lambda x: {k: round(v, 3) for k, v in x.items()} if x else '-'
            print(f'{tag}  probe {fmt(lp)}  gen {fmt(ev)}')


007_p2_COLAB_FULL_B05_sft_only_seed0  probe {'easy': 0.688, 'medium': 0.522, 'hard': 0.351}  gen {'easy': 0.518, 'medium': 0.432, 'hard': 0.29}
007_p2_COLAB_FULL_B05_sft_only_seed1  probe {'easy': 0.684, 'medium': 0.511, 'hard': 0.364}  gen -
007_p2_COLAB_FULL_B05_sft_only_seed2  probe {'easy': 0.691, 'medium': 0.519, 'hard': 0.374}  gen -
007_p2_COLAB_FULL_B3_sft_only_seed0  probe {'easy': 0.86, 'medium': 0.729, 'hard': 0.495}  gen {'easy': 0.857, 'medium': 0.701, 'hard': 0.467}
007_p2_COLAB_FULL_B3_sft_only_seed1  probe {'easy': 0.859, 'medium': 0.726, 'hard': 0.503}  gen -
007_p2_COLAB_FULL_B3_sft_only_seed2  probe {'easy': 0.861, 'medium': 0.731, 'hard': 0.492}  gen -
007_p8_COLAB_FULL_B05_sft_only_seed0  probe {'easy': 0.673, 'medium': 0.511, 'hard': 0.354}  gen {'easy': 0.667, 'medium': 0.461, 'hard': 0.371}
007_p8_COLAB_FULL_B05_sft_only_seed1  probe {'easy': 0.679, 'medium': 0.522, 'hard': 0.351}  gen -
007_p8_COLAB_FULL_B05_sft_only_seed2  probe {'easy': 0.678, 'medium': 0.509

In [ ]:
# closing balance, completes the open record for this session
cu_after = 493.07   # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)


reminder: fill cu_after for the record
